# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [1]:
# Fill in your information
name = "Kawsar Jamil"           # Write your full name here
email = "Kawsarjamil726@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Kawsar Jamil
Email : Kawsarjamil726@gmail.com


---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler,OneHotEncoder,RobustScaler
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

In [3]:
# Question 1
df=pd.read_csv("house_price_regression_dataset.csv")
print(df.shape)
print(df.head(10))
print('\n')
df.sample(5)


(1000, 8)
   Square_Footage  Num_Bedrooms  Num_Bathrooms  Year_Built  Lot_Size  \
0            1360             2              1        1981  0.599637   
1            4272             3              3        2016  4.753014   
2            3592             1              2        2016  3.634823   
3             966             1              2        1977  2.730667   
4            4926             2              1        1993  4.699073   
5            3944             5              3        1990  2.475930   
6            3671             1              2        2012  4.911960   
7            3419             1              1        1972  2.805281   
8             630             3              3        1997  1.014286   
9            2185             4              2        1981  3.941604   

   Garage_Size  Neighborhood_Quality   House_Price  
0            0                     5  2.623829e+05  
1            1                     6  9.852609e+05  
2            0                     9  

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
903,4100,2,1,1956,1.952270,1,1,809712.737933
248,516,5,3,1959,3.575542,1,1,171023.956598
298,4347,3,1,1966,4.231914,1,3,895675.554821
632,1399,2,2,2003,1.680543,0,3,310271.263669
882,2145,4,1,2009,4.659292,2,1,524818.963329


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [4]:
df=df[df.columns[df.isna().mean()<=0.5]]

x=df.drop(columns=["House_Price"])
y=df["House_Price"]
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.20,random_state=42
)

numeric_feature=x.select_dtypes(
    include=np.number).columns.tolist()

numeric_feature=[col for col in numeric_feature if col!="Neighborhood_Quality"]

# print(numeric_feature)

cat_feature=["Neighborhood_Quality"]
# cat_feature

num_imputer=SimpleImputer(missing_values=np.nan, strategy="mean")
num_imputer.fit(x_train[numeric_feature])
x_train[numeric_feature]=num_imputer.transform(x_train[numeric_feature])
x_test[numeric_feature]=num_imputer.transform(x_test[numeric_feature])

cat_imputer=SimpleImputer(missing_values=np.nan,strategy="most_frequent")

cat_imputer.fit(x_train[cat_feature])
x_train[cat_feature]=cat_imputer.transform(x_train[cat_feature])
x_test[cat_feature]=cat_imputer.transform(x_test[cat_feature])




print(x_train.shape,y_train.shape)
print('\n')
print(x_test.shape,y_test.shape)

(800, 7) (800,)


(200, 7) (200,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [5]:
scaler=StandardScaler()
scaler.fit(x_train[["Square_Footage"]])
x_train["Square_Footage"]=scaler.transform(x_train[["Square_Footage"]]).ravel()
x_test["Square_Footage"]=scaler.transform(x_test[["Square_Footage"]]).ravel()
def make_pred(x, w, b):
    m = x.shape[0]
    prediction = np.zeros(m)

    for i in range(m):
        prediction[i] = w * x.iloc[i] + b
    return prediction


def compute_cost(x, y, w, b):
    m = x.shape[0]
    cost = 0

    for i in range(m):
        prediction = w * x.iloc[i] + b
        error = prediction - y.iloc[i]
        error_sqrd = error**2
        cost = cost + error_sqrd

    cost = cost / (2 * m)
    return cost


def calc_gradient(x, y, w, b):
    m = x.shape[0]

    dj_dw = 0
    dj_db = 0

    for i in range(m):
        prediction = w * x.iloc[i] + b
        error = prediction - y.iloc[i]

        dj_dw = dj_dw + (error * x.iloc[i])
        dj_db = dj_db + error

    return dj_dw / m, dj_db / m


def gradient_descent(x, y, w_in, b_in,alpha, max_iter=1000):
    w = w_in
    b = b_in
    for i in range(max_iter):
        dj_dw, dj_db = calc_gradient(x, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        cost = compute_cost(x, y, w,b)
    return w, b


w_init = 0
b_init = 0
w_final, b_final = gradient_descent(
    x_train["Square_Footage"],
    y_train,
    w_in=w_init,
    b_in=b_init,
    max_iter=1000,
    alpha=0.01,
)
print(w_final, b_final)

prediction = make_pred(x_test["Square_Footage"], w_final, b_final)
for i in range(10):
    print(f"pred:{prediction[i]:.4f} real value: {y_test.iloc[i]:.4f}\n")

251083.68779527766 618549.3497204983
pred:858826.1704 real value: 901000.4882

pred:517493.1024 real value: 494537.5115

pred:998407.7305 real value: 949404.2007

pred:1043330.5315 real value: 1040389.0481

pred:785425.5224 real value: 794010.0167

pred:773392.6293 real value: 724033.5605

pred:984369.3552 real value: 998439.2371

pred:881889.2155 real value: 909713.4350

pred:812499.5319 real value: 792681.5184

pred:905152.8089 real value: 947490.7775



---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [6]:
#Question 4

preprocessor=ColumnTransformer(
    transformers=[
        ("scale",StandardScaler(),["Square_Footage","Num_Bedrooms","Num_Bathrooms"]),
        ("encode",OneHotEncoder(sparse_output=False, drop='first'),["Neighborhood_Quality"])
    ],
    verbose_feature_names_out=False
).set_output(transform='pandas')

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [7]:
# Question 5
sgd_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            SGDRegressor(
                penalty=None,
                max_iter=1000000,
                learning_rate="adaptive",
                eta0=0.01,
                random_state=42,
            ),
        ),
    ]
)

sgd_pipeline.fit(x_train, y_train)
prediction = sgd_pipeline.predict(x_test)

print("R2  :", round(r2_score(y_test, prediction), 4))
print("RMSE :", round(np.sqrt(mean_squared_error(y_test, prediction)), 4), "\n")


for i in range(10):
    print(f"predict:{prediction[i]:.4f}, actual:{y_test.iloc[i]:.4f}\n")

R2  : 0.9869
RMSE     : 29097.7138 

predict:851442.5994, actual:901000.4882

predict:507679.2702, actual:494537.5115

predict:986028.0834, actual:949404.2007

predict:1022692.3254, actual:1040389.0481

predict:760739.1345, actual:794010.0167

predict:765811.3928, actual:724033.5605

predict:1005091.1617, actual:998439.2371

predict:903066.0644, actual:909713.4350

predict:798013.5940, actual:792681.5184

predict:888886.8342, actual:947490.7775



---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [8]:
# Question 6
lr_pipeline=Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("model",LinearRegression())
    ]
)

lr_pipeline.fit(x_train,y_train)
prediction=lr_pipeline.predict(x_test)

print("R2  :", round(r2_score(y_test, prediction),4))
print("RMSE     :", round(np.sqrt(mean_squared_error(y_test,prediction)),4),'\n')


for i in range(10):
    print(f"predict:{prediction[i]:.4f}, actual:{y_test.iloc[i]:.4f}\n")

R2  : 0.9869
RMSE     : 29093.0752 

predict:851409.1473, actual:901000.4882

predict:507641.0339, actual:494537.5115

predict:985990.3750, actual:949404.2007

predict:1022654.7065, actual:1040389.0481

predict:760698.1337, actual:794010.0167

predict:765782.3164, actual:724033.5605

predict:1005039.0963, actual:998439.2371

predict:903407.7818, actual:909713.4350

predict:797971.1884, actual:792681.5184

predict:888850.4300, actual:947490.7775



---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [31]:
# Question 7
features = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
x_train_mat = x_train[features].sample(100,random_state=42)

y_train_mat = y_train[x_train_mat.index].to_numpy()
x_train_mat = x_train_mat.to_numpy()
one_vector = np.ones(x_train_mat.shape[0]).reshape(-1, 1)
x_train_mat_mod = np.concatenate((one_vector, x_train_mat), axis=1)
theta = np.dot(
    np.linalg.pinv(np.dot(x_train_mat_mod.T, x_train_mat_mod)),
    (np.dot(x_train_mat_mod.T, y_train_mat)),
)
print(f"bias: {theta[0]}\nweights: {theta[1:]}")


bias: 566455.9051499707
weights: [251170.08732673   9871.17161559   9515.815829  ]
